<a href="https://colab.research.google.com/github/henriquevoigt/Treinamento-H2IA/blob/main/H2IA_W5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# O Problema
Sliding Puzzle - Bloco Deslizante

In [17]:
from IPython.display import Image
Image(url='https://miro.medium.com/max/700/1*W7jg4GmEjGBypd9WPktasQ.gif',width=200)

# Resolver o quebra-cabeças usando Buscas

## Estrutura

In [18]:
# tupla de tuplas 3x3
OBJETIVO = (
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 0)
)

POSICAO_CORRETA = {
    1: (0, 0), 2: (0, 1), 3: (0, 2),
    4: (1, 0), 5: (1, 1), 6: (1, 2),
    7: (2, 0), 8: (2, 1)
}

class No:
    def __init__(self, estado, pai=None, acao=None, g=0, h=0):
        self.estado = estado
        self.pai = pai
        self.acao = acao
        self.g = g
        self.h = h
        self.f = g + h

    def __lt__(self, outro):
        return self.f < outro.f

def eh_soluvel(estado):

    lista_plana = []
    for linha in estado:
        for valor in linha:
            if valor != 0:
                lista_plana.append(valor)

    inversoes = 0

    for i in range(len(lista_plana)):
        for j in range(i + 1, len(lista_plana)):
            if lista_plana[i] > lista_plana[j]:
                inversoes += 1

    return inversoes % 2 == 0

def encontrar_zero(estado):
    for i in range(3):
        for j in range(3):
            if estado[i][j] == 0:
                return i, j

def gerar_filhos(no):
    filhos = []
    linha, coluna = encontrar_zero(no.estado)

    acoes = [
        ("CIMA", -1, 0),
        ("BAIXO", 1, 0),
        ("ESQUERDA", 0, -1),
        ("DIREITA", 0, 1)
    ]

    for nome_acao, delta_linha, delta_coluna in acoes:
        nova_linha = linha + delta_linha
        nova_coluna = coluna + delta_coluna

        if 0 <= nova_linha < 3 and 0 <= nova_coluna < 3:

            # converte a tupla atual em lista de listas para poder alterar
            novo_estado = [list(row) for row in no.estado]

            novo_estado[linha][coluna], novo_estado[nova_linha][nova_coluna] = novo_estado[nova_linha][nova_coluna], novo_estado[linha][coluna]

            # converte a lista de listas de volta para tupla de tuplas
            estado_final_lista = []
            for row in novo_estado:
                estado_final_lista.append(tuple(row)) # converte cada linha em tupla
            estado_final_tupla = tuple(estado_final_lista) # converte o bloco todo em tupla

            novo_g = no.g + 1
            novo_h = calcula_heuristica(estado_final_tupla)

            filhos.append(No(estado_final_tupla, no, nome_acao, novo_g, novo_h))

    return filhos

def reconstituir_caminho(no):
    caminho = []
    while no:
        caminho.append((no.estado, no.acao))
        no = no.pai
    caminho.reverse()
    return caminho

def calcula_heuristica(estado):
    h = 0
    for i in range(3):
        for j in range(3):
            valor = estado[i][j]
            if valor != 0:
                linha_correta, coluna_correta = POSICAO_CORRETA[valor]
                h += abs(i - linha_correta) + abs(j - coluna_correta)
    return h

## A*


In [19]:
import heapq

def busca_a_estrela(estado_inicial):
    h_inicial = calcula_heuristica(estado_inicial)
    no_inicial = No(estado_inicial, pai=None, acao=None, g=0, h=h_inicial)

    fronteira = []

    heapq.heappush(fronteira, no_inicial)

    visitados = set()
    nos_expandidos = 0

    while len(fronteira) > 0:

        no_atual = heapq.heappop(fronteira)

        if no_atual.estado in visitados:
            continue

        visitados.add(no_atual.estado)
        nos_expandidos += 1

        if no_atual.estado == OBJETIVO:
            print(f"Nós Expandidos (Tempo): {nos_expandidos}")
            print(f"Fronteira (Memória): {len(fronteira)}")
            return reconstituir_caminho(no_atual)

        filhos_gerados = gerar_filhos(no_atual)

        for filho in filhos_gerados:
            if filho.estado not in visitados:
                heapq.heappush(fronteira, filho)

    return None

## Teste A*

In [20]:
import time

tabuleiro_teste = (
    (8, 2, 3),
    (1, 6, 4),
    (7, 0, 5)
)

if eh_soluvel(tabuleiro_teste):
    print("O tabuleiro tem solução! Iniciando A*...")
    inicio = time.time()
    caminho_solucao_astar = busca_a_estrela(tabuleiro_teste)
    fim = time.time()

    tempo_execucao = fim - inicio
    print(f"Tempo de execução do algoritmo: {tempo_execucao:.5f} segundos")

    print(f"Solução encontrada em {len(caminho_solucao_astar) - 1} movimentos\n")

    '''
    for passo, (estado, acao) in enumerate(caminho_solucao_astar):
      if acao is None:
        print("Estado inicial:")
      else:
        print(f"Passo {passo} - Movimento: {acao}")

      for linha in estado:
        print(linha)
      print("-" * 20)
      '''
else:
    print("ERRO: Este tabuleiro é matematicamente insolúvel. Operação abortada para poupar recursos.")

O tabuleiro tem solução! Iniciando A*...
Nós Expandidos (Tempo): 89
Fronteira (Memória): 63
Tempo de execução do algoritmo: 0.00182 segundos
Solução encontrada em 15 movimentos



## Discorra sobre o desempenho do método comparando com as buscas cegas da semana anterior:




O uso de A* mudou drasticamente a eficiência, mantendo a otimalidade matemática do BFS e encontrando os mesmos 15 movimentos, mas expandindo apenas 89 nós. Uma redução gigante no esforço computacional em comparação aos 6.575 nós da busca cega. Essa eficiência ocorre porque a heurística atua como um "farol", em vez de explorar o tabuleiro "no cego", o cálculo heurístico poda as ramificações inúteis e direciona a busca apenas para os estados que de fato aproximam as peças de suas posições corretas, mantendo a fronteira enxuta.

Desafio Extra: A vantagem de checar a solubilidade antes é poupar recurso computacional e garantir que só o gastaremos se for solúvel. Ao tentar resolver um estado insolúvel ficaremos presos em uma busca exaustiva, estourando o tempo de execução. Como não há solução, varreremos todos os estados possíveis. Em versões maiores o problema é maior ainda, uma vez que a explosão combinatória é maior.